In [1]:
import numpy as np
import torch
import scanpy as sc
import anndata as ad
import os
import pandas as pd
from utils.preprocess import *

In [2]:
import os
import sys
    
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch
import torchsde
from torchdyn.core import NeuralODE
from tqdm import tqdm

from torchcfm.conditional_flow_matching import *
from torchcfm.models import MLP
from torchcfm.utils import plot_trajectories, torch_wrapper
from simulate.simulate import *
from omegaconf import OmegaConf
from utils.hydra import *
from datasets.process import *
from scripts.run_model import *
from eval.eval import *

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
%reload_ext autoreload
%autoreload 2

In [5]:
############################################################

In [6]:
config = load_config(overrides=['dataset=cite'])
OmegaConf.set_struct(config, False)
config.classifier.num_layers = 2
config.classifier.hidden_dim = 256
config.classifier.epsilon = 0.05

In [7]:
### SETTINGS ###

adata = process_data(pc_dim=config.pc_dim, 
                     t0_index=config.t0_index,
                     t1_index=config.t1_index,
                     data=config.dataset, 
                     use_paga=config.use_paga,
                     tissue=config.tissue)

print(adata.obs['cell_type'].nunique())

timepoints = sorted(adata.obs['timepoint'].unique().tolist())
tree = adata.uns['tree']

config.num_classes = adata.obs['cell_type'].nunique()

print(config.t0_index)
print(config.t1_index)

/home/azweig/projects/finfm/utils/lineage.py:174: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns['stoi'] = stoi


7
1
3


In [8]:
t0, t1 = timepoints[config.t0_index], timepoints[config.t1_index]

adata = adata[(adata.obs['timepoint'] >= t0) & (adata.obs['timepoint'] <= t1)]
adata_train = adata[adata.obs['timepoint'].isin([t0, t1])]

print(t0, t1)

3 7


In [9]:
# # CFM
# config.metric = "cfm"
# config.no_learning = True

In [10]:
# MFM
config.metric = "mfm"
config.mfm.use_euclidean_ot = True
config.mfm.K = 150
config.mfm.kappa = 1.5
config.mfm.epsilon = 1e-2

# config.finsler.use = True
# config.finsler.lamb = 3.0
# config.dummy_kl_weight = 0.1

config.classifier_max_epochs = 2
config.metric_max_epochs = 5000


In [11]:
# adata.obs['donor'].unique()

In [12]:
############################################################

In [13]:
# from scripts.run_gaga import *
# classifier_model, ae_model, disc_model, metric_model, embed_model = run_gaga_model(config, 
#                                                                                    config.project, 
#                                                                                    adata_train, 
#                                                                                    timepoints, 
#                                                                                    tree, 
#                                                                                    wandb_logger=None)

In [14]:
singleton_dataloader = build_singleton_dataloader(config, adata_train)
paired_dataloader = build_paired_dataloader(config, adata_train)

In [15]:

classifier_model, metric_model, embed_model, flow_model = run_full_model(config=config,
                                                                         project=config.project,
                                                                         singleton_dataloader=singleton_dataloader,
                                                                         paired_dataloader=paired_dataloader,
                                                                         timepoints=timepoints,
                                                                         tree=tree)

{'pc_dim': 100, 'num_classes': 7, 'seed': 42, 'classifier_max_epochs': 2, 'metric_max_epochs': 5000, 'embed_max_epochs': 3000, 'flow_max_epochs': 2, 'gradient_clip_val': 0, 'warmup_steps': 0, 'loader_batch_size': 6, 'flow_batch_size': 512, 'score_batch_size': 2048, 'ema_decay': '1-1e-3', 'use_paga': False, 'paga_threshold': 0.0, 'lr': 0.001, 'dropout': 0.0, 'hidden_dim': 256, 'latent_dim': 100, 'num_layers': 3, 'num_freq': 32, 'rescale': 0.5, 'skip': False, 'normalize': True, 'balance_classes': False, 'force_cpu': False, 'detect_anomaly': False, 'use_wandb': True, 'dummy_prior': 0.05, 'dummy_kl_weight': 0.05, 'classifier': {'hidden_dim': 256, 'num_layers': 2, 'spectral_norm': False, 'epsilon': 0.05}, 'metric': 'mfm', 'sigma': 0.0, 'ot_in_embed': True, 'fast_ot': False, 'no_learning': False, 'time_per_batch': 4, 'random_velocities': False, 'method': 'exact', 'reg': -1, 'reg_m': -1, 'mfm': {'K': 150, 'kappa': 1.5, 'alpha': 1.0, 'epsilon': 0.01, 'use_euclidean_ot': True}, 'finsler': {'use

wandb: Currently logged in as: az831 (az831-new-york-genome-center) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
You are using a CUDA device ('NVIDIA GeForce RTX 4080 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type           | Params | Mode 
----------------------------------------------------

epoch,▁▁▁▁▁▁██████
train_ce,█▄▃▃▂▂▂▂▁▁▁▁
train_kl,▁▂▄▆▆▇▇▇▇███
train_loss_classifier,█▃▃▃▂▂▂▂▁▁▁▁
trainer/global_step,▁▂▂▃▄▄▅▅▆▇▇█
epoch,1
train_ce,1.33825
train_kl,1.18264
train_loss_classifier,1.39738
trainer/global_step,11


Running phase metric:.......


...

epoch,▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇████
train_loss_metric,██▇▇▇▆▆▆▅▅▅▅▄▄▄▄▄▄▄▄▄▄▄▄▃▃▃▂▂▃▂▂▂▂▁▁▁▁▁▁
trainer/global_step,▁▁▁▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇█
epoch,4999
train_loss_metric,6809.50586
trainer/global_step,29999


Running phase embed:.......


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type           | Params | Mode 
--------------------------------------------------------
0 | embed_net    | SimpleEmbedNet | 184 K  | train
1 | geo_net      | SinNet         | 226 K  | train
2 | metric_model | MetricNetMFM   | 150    | eval 
--------------------------------------------------------
411 K     Trainable params
150       Non-trainable params
411 K     Total params
1.646     Total estimated model params size (MB)
26        Modules in train 

Restored best model from epoch 2819 with train_loss_geo_embed=2.079778


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇█████
train_loss_embed,█▅▂▂▂▁▁▂▂▁▁▁▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▃▁▁▂▁▁▂▁▁▁▁▁▁
train_loss_geo,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss_geo_embed,▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▂▁▁▁█▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████
epoch,2999
train_loss_embed,1.13278
train_loss_geo,1.31511
train_loss_geo_embed,2.44789
trainer/global_step,2999


In [16]:
#fix the wandb.run.summary bug?
def remove_all_forward_hooks(model):
    for module in model.modules():
        module._forward_hooks.clear()

remove_all_forward_hooks(classifier_model)
remove_all_forward_hooks(metric_model)
remove_all_forward_hooks(embed_model)

In [17]:
t0, t1 = timepoints[config.t0_index], timepoints[config.t1_index]

w1_scores = []
for index in range(config.t0_index + 1, config.t1_index):
    print(index)
    t = timepoints[index]
    w1 = predict(embed_model, adata, t0, t, t1, num_traj=6000, library="pot")
    w1_scores.append(w1)
w1_scores = torch.tensor(w1_scores)
print(w1_scores)
print(torch.mean(w1_scores))

2


/home/azweig/projects/finfm/models/embed_models.py:135: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t = torch.tensor(j)


tensor([41.7802])
tensor(41.7802)


/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/ot/lp/_network_simplex.py:574: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  check_result(result_code)
